In [3]:
# IMPORT LIBRARIES
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
# IMPORT SRC
from src.data.dataset import TextDataset
from src.tokenizer.bpe_tokenizer import BPETokenizer
from src.models.model import TransformerLM
from src.models.base_model import Trainer
from src.utils.config import load_config

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
config_path = 'config.yaml'
CONFIG = load_config(config_path)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# ── Data ──────────────────────────────────────────────────────────────────────
with open('data/data.txt', 'r') as f:
    data = f.read()
CONTEXT_LEN = CONFIG.model.context_len

In [5]:
#  ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = BPETokenizer(num_merges=100)
tokenizer.train(data)
tokenized_data = tokenizer.encode(data)
tokenized_data = torch.tensor(tokenized_data, dtype=torch.long)

#  ── Dataset ───────────────────────────────────────────────────────────────────
dataset = TextDataset(tokenized_data, context_len=CONFIG.model.context_len)
loader  = DataLoader(dataset, batch_size=CONFIG.training.batch_size, shuffle=True)

#  ── Training ──────────────────────────────────────────────────────────────────
model     = TransformerLM(tokenizer.vocab_size, CONFIG.model).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.training.lr)

trainer = Trainer(DEVICE, model, optimizer, loader)
trainer.train(CONFIG.training.steps)

step     0  loss 4.9845
step   500  loss 0.0561
step  1000  loss 0.0433


In [6]:
# ── Generate ──────────────────────────────────────────────────────────────────
def generate(tokenized_prompt: str, n_tokens: int = 200, temperature: float = 1.0) -> str:
    model.eval()
    ctx = tokenized_prompt[-CONTEXT_LEN:]  # encode prompt, take last window
    out = []

    with torch.no_grad():
        for _ in range(n_tokens):
            x = torch.tensor([ctx], dtype=torch.long, device=DEVICE)
            logits = model(x)
            logits = logits[0, -1] / (temperature + 1e-6)
            probs  = F.softmax(logits, dim=-1)
            next_t = torch.multinomial(probs, 1).item()
            out.append(next_t)
            ctx = ctx[1:] + [next_t]   # slide window
    return out

In [21]:
prompt = 'wh a pear' + '\n'
print('Prompt:')
print(prompt)
print('Answer:')
print(tokenizer.decode(generate(tokenizer.encode(prompt), n_tokens=300, temperature=1)))

Prompt:
wh a pear

Answer:
The ter lime (Citrus australasica), blood lime (hybrid), and desert lime among others. Limes are a rich source of vitamin C and are used to accent the flavours of foods and beverages. In 2023, world production of limes (combined with lemons) was 23.6 million tropical and subtropical regions for culinary, medicinal, and ornamental purposes.

The term "lime" is used for a variety of citrus fruits, including the Key lime (Citrus × aurantiifolia), Persian lime (Citrus × latifolia), Makrut lime (Citrus hystrix), finger lime (Citrus australasica), blood lime (hybrid), and desert lime among others. Limes are a rich source of vitamin C and are used to a
